In [ ]:
# --- 1. Install gdown for Google Drive Downloads ---
# We use -q to make the output "quiet"
print("Installing gdown...")
!pip install gdown -q

# --- 2. Import All Necessary Libraries ---
import os
import gdown  # For downloading from Google Drive
import torch
import torch.nn as nn
import timm
import matplotlib.pyplot as plt
from google.colab import files
from PIL import Image
import io
from torchvision import transforms
from collections import OrderedDict
from google.colab import drive # Import drive module

print("--- Interactive Prediction Cell (v9 - GDrive Link) ---")

# --- 3. Define the Model Architecture ---
# This class MUST match the one used during training.
# Based on previous errors, the saved weights are for an attribute named 'model'.
class UltraSimpleModel(nn.Module):
    def __init__(self, pretrained=False):
        super(UltraSimpleModel, self).__init__()
        # This attribute name 'model' MUST match the saved keys
        self.model = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=1)

    def forward(self, x_dict):
        # The model expects a dictionary input
        rgb_input = x_dict['rgb']
        # We call self.model here
        output = self.model(rgb_input)
        return output

# --- 4. Define Paths and Download Model ---
MODEL_PATH = "final_model_complete.pth"
FILE_ID = "1MPxXJ7ghdZ8XafhuNUXqjNPJyTrg1zhE" # ID from your shared link

if not os.path.exists(MODEL_PATH):
    print(f"Downloading model file: {MODEL_PATH}...")
    try:
        # Download the file from the public Google Drive link
        gdown.download(id=FILE_ID, output=MODEL_PATH, quiet=False)
        print("Model downloaded successfully.")
    except Exception as e:
        print(f"Error downloading model: {e}")
        print("Please ensure the Google Drive link is correct and publicly accessible.")
else:
    print(f"Model file '{MODEL_PATH}' already exists locally.")

# --- 5. Define Image Transforms ---
# These MUST be the same as the validation transforms used during training
IMAGE_SIZE = (256, 256)
val_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# --- 6. Load the Trained Model ---
print(f"Loading model from: {MODEL_PATH}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UltraSimpleModel(pretrained=False) # Create an instance of our (now correct) model class

try:
    # Load the checkpoint dictionary
    checkpoint = torch.load(MODEL_PATH, map_location=device)

    # Access the saved model weights, which are stored under the 'model_state_dict' key
    model.load_state_dict(checkpoint['model_state_dict'])

    model.to(device)
    model.eval() # Set model to evaluation mode
    print("Model loaded successfully and set to evaluation mode.")

    # --- 7. Uploader and Predictor ---
    print("\nPlease upload an image to classify (AI or Real)...")
    uploaded = files.upload()

    if len(uploaded) == 0:
        print("No file uploaded. Please run the cell again to try.")
    else:
        # Get the first uploaded file
        file_name = list(uploaded.keys())[0]
        img_bytes = uploaded[file_name]

        try:
            # Open image and display
            img_pil = Image.open(io.BytesIO(img_bytes)).convert('RGB')

            # Preprocess for the model
            # .unsqueeze(0) adds the batch dimension (batch size = 1)
            tensor = val_transforms(img_pil).unsqueeze(0).to(device)

            # Create the dictionary input the model expects
            input_dict = {'rgb': tensor}

            # Get prediction
            with torch.no_grad(): # Disable gradient calculation for inference
                output = model(input_dict)
                prob = torch.sigmoid(output).item() # Probability (0.0 to 1.0)

            # Determine predicted label and confidence
            if prob > 0.5:
                pred_label = 'AI'
                confidence = prob
                color = 'red'
            else:
                pred_label = 'Real'
                confidence = 1 - prob
                color = 'green'

            # Display the image with the prediction
            plt.figure(figsize=(6, 6))
            plt.imshow(img_pil)
            plt.title(f"Prediction: {pred_label}\nConfidence: {confidence:.2%}", color=color, fontsize=16, pad=10)
            plt.axis('off')
            plt.show()

        except Exception as e:
            print(f"Error processing image: {e}")
            print("Please make sure you uploaded a valid image file (e.g., .jpg, .png).")

except FileNotFoundError:
    print(f"ERROR: Model file not found at {MODEL_PATH}")
    print("Please ensure the path is correct or the download was successful.")
except RuntimeError as e:
    print(f"An error occurred while loading the model: {e}")
    print("This error often means the model class definition above does not match the saved file.")
except Exception as e:
    print(f"An unknown error occurred during model loading: {e}")